In [1]:

# imports
import os
import sys
import types
import json

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = fig_dpi
  from IPython.display import set_matplotlib_formats
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  pio.renderers.default = "notebook_connected"
except Exception:
  pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass



# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
if r'G:\wjonasreger.github.io\projects\text_classification_neural_networks':
  os.chdir(r'G:\wjonasreger.github.io\projects\text_classification_neural_networks')

# reset state
%reset

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v
  
  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define


C:\Users\jonas\AppData\Local\Temp/ipykernel_42740/638849703.py:20: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats(fig_format)


{"C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\importlib\\_bootstrap.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\importlib\\_bootstrap_external.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\codecs.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\encodings\\aliases.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\encodings\\__init__.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\encodings\\utf_8.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\encodings\\latin_1.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\abc.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\io.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\stat.p

In [2]:
#| eval: false
import random
from collections import defaultdict
from tqdm.notebook import tqdm

import torch
import torchtext
from torch.utils import data

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

random.seed(22)

In [3]:
#| eval: false
# set device to cuda for faster model training
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

In [4]:
#| eval: false
def preprocess(review):
  res = []
  for x in review.split(' '):
    remove_beg=True if x[0] in {'(', '"', "'"} else False
    remove_end=True if x[-1] in {'.', ',', ';', ':', '?', '!', '"', "'", ')'} else False
    if remove_beg and remove_end: res += [x[0], x[1:-1], x[-1]]
    elif remove_beg: res += [x[0], x[1:]]
    elif remove_end: res += [x[:-1], x[-1]]
    else: res += [x]
  return res

train_data = torchtext.datasets.IMDB(root='.data', split='train')
train_data = list(train_data)
train_data = [(x[0], preprocess(x[1])) for x in train_data]
train_data, test_data = train_data[0:10000] + train_data[12500:12500+10000], train_data[10000:12500] + train_data[12500+10000:], 

print('Num. Train Examples:', len(train_data))
print('Num. Test Examples:', len(test_data))

In [5]:
#| eval: false
print("\nSAMPLE DATA:")
for x in random.sample(train_data, 5):
  print('Sample label:', x[0])
  print('Sample text:', ' '.join(x[1]), '\n')

In [6]:
#| eval: false
PAD = '<PAD>'
END = '<END>'
UNK = '<UNK>'

In [7]:
#| eval: false
class TextDataset(data.Dataset):
  def __init__(self, examples, split, threshold, max_len, idx2word=None, word2idx=None):
    self.examples = examples
    assert split in {'train', 'val', 'test'}
    self.split = split
    self.threshold = threshold
    self.max_len = max_len

    # dictionaries
    self.idx2word = idx2word
    self.word2idx = word2idx
    if split == 'train':
      self.build_dictionary()
    self.vocab_size = len(self.word2idx)
    
    # convert text to indices
    self.textual_ids = []
    self.convert_text()
  
  def build_dictionary(self): 
    assert self.split == 'train'
    self.idx2word = {0:PAD, 1:END, 2: UNK}
    self.word2idx = {PAD:0, END:1, UNK: 2}

    # build dictionaries
    word_freq = defaultdict(float)
    for review in self.examples:
      text = review[1] # text of review
      for word in text:
        word_freq[word.lower()] += 1 # increment word frequency (lower cased)
    for word in word_freq.keys():
      if word_freq[word] >= self.threshold: # add new word and id to dictionaries once threshold is exceeded
        new_id = len(self.idx2word)
        self.idx2word[new_id] = word
        self.word2idx[word] = new_id
  
  def convert_text(self):
    for review in self.examples:
      label = review[0]
      text = review[1]
      converted_text = []

      # text conversion
      converted_text = list(map(lambda word: self.word2idx[word] if word in self.word2idx.keys() else self.word2idx[UNK], text))
      converted_text.append(self.word2idx[END])
      self.textual_ids.append((label, converted_text))

  def get_text(self, idx):
    text = self.textual_ids[idx][1]

    # pad or truncate the review as needed
    updated_text = text[:self.max_len] if len(text) >= self.max_len else text + [0]*(self.max_len-len(text))
    updated_text = torch.tensor(updated_text)

    return updated_text
  
  def get_label(self, idx):
    label = self.textual_ids[idx][0]
    score = 1 if label == 'pos' else 0
    score = torch.tensor(score) # sentiment score of the review as a tensor

    return score

  def __len__(self):
    num = len(self.examples) # number of reviews

    return num
  
  def __getitem__(self, idx):
    # get the processed text and score of the review
    review, label = self.get_text(idx), self.get_label(idx)

    return review, label

In [8]:
#| eval: false
train_dataset = TextDataset(train_data, 'train', threshold=10, max_len=150)
print('Vocab size:', train_dataset.vocab_size, '\n')

randidx = random.randint(0, len(train_dataset)-1)
text, label = train_dataset[randidx]
print('Example text:')
print(train_data[randidx][1])
print(text)
print('\nExample label:')
print(train_data[randidx][0])
print(label)

In [9]:
#| eval: false
class CNN(nn.Module):
  def __init__(self, vocab_size, embed_size, out_channels, filter_heights, stride, dropout, num_classes, pad_idx):
    super(CNN, self).__init__()

    # embedding layer
    self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_size, padding_idx=pad_idx)
    # multiple convolution layers
    self.conv_layers = nn.ModuleList( [nn.Conv2d(in_channels=1, out_channels=out_channels, kernel_size=[filter_height, embed_size]) for filter_height in filter_heights] )
    # dropout layer
    self.dropout = nn.Dropout(dropout)
    # linear layer
    self.linear = nn.Linear(out_channels * len(filter_heights), num_classes)

  def forward(self, texts):
    # embed texts -- [batch_size, max_len, embed_size]
    x = self.embedding(texts)
    # modify embedded output dimensions -- [batch_size, 1, MAX_LEN, embed_size] (i.e., unsqueeze)
    x = x.unsqueeze(1)
    # pass embedded texts into cnn layers -- [batch_size, out_channels, *, 1]
    # modify dimension of outputs -- [batch_size, out_channels, *] (i.e., squeeze)
    # apply non-linearity
    x = [ F.relu( conv_layer(x) ).squeeze(3) for conv_layer in self.conv_layers ]
    # get max value across last dimension via pooling -- [batch_size, out_channels]
    x = [ F.max_pool1d( value, value.size(2) ).squeeze(2) for value in x ]
    # concatenate outputs from cnn layers -- [batch_size, (out_channels*num_of_cnn_layers)]
    x = torch.cat(x, 1)
    # apply dropout
    x = self.dropout(x)
    # pass output through linear layer -- [batch_size, num_classes]
    x = self.linear(x)

    # a softmax is applied later in model training
    return x

In [10]:
#| eval: false
# helper function to count number of parameters in a model
count_parameters = lambda model: sum(p.numel() for p in model.parameters() if p.requires_grad)

In [11]:
#| eval: false
# adjustable parameters (defaults: THRESHOLD=5, MAX_LEN=200, BATCH_SIZE=32)
THRESHOLD = 5
MAX_LEN = 200
BATCH_SIZE = 32

train_dataset = TextDataset(train_data, 'train', THRESHOLD, MAX_LEN)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
print('train vocab size:', train_dataset.vocab_size)

test_dataset = TextDataset(test_data, 'test', THRESHOLD, MAX_LEN, train_dataset.idx2word, train_dataset.word2idx)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=1, drop_last=False)
print('test vocab size:', test_dataset.vocab_size)

In [12]:
#| eval: false
def train_model(model, num_epochs, data_loader, optimizer, criterion):
  print('Training Model...')
  model.train()
  for epoch in tqdm(range(num_epochs)):
    epoch_loss = 0
    epoch_acc = 0
    for texts, labels in data_loader:
      texts = texts.to(DEVICE) # shape: [batch_size, MAX_LEN]
      labels = labels.to(DEVICE) # shape: [batch_size]

      optimizer.zero_grad()

      output = model(texts)
      acc = accuracy(output, labels)
      
      loss = criterion(output, labels)
      loss.backward()
      optimizer.step()

      epoch_loss += loss.item()
      epoch_acc += acc.item()
    print('[TRAIN]\t Epoch: {:2d}\t Loss: {:.4f}\t Train Accuracy: {:.2f}%'.format(epoch+1, epoch_loss/len(data_loader), 100*epoch_acc/len(data_loader)))
  print('Model Trained!\n')

In [13]:
#| eval: false
def accuracy(output, labels):
  preds = output.argmax(dim=1) # find predicted class
  correct = (preds == labels).sum().float() # convert into float for division 
  acc = correct / len(labels)
  return acc

In [14]:
#| eval: false
cnn_model = CNN(vocab_size = train_dataset.vocab_size,
            embed_size = 128, 
            out_channels = 64, 
            filter_heights = [2, 3, 4], 
            stride = 1, 
            dropout = 0.5, 
            num_classes = 2,
            pad_idx = train_dataset.word2idx[PAD])

# load the model on the device (cuda or cpu)
cnn_model = cnn_model.to(DEVICE)

print('The model has {:,d} trainable parameters'.format(count_parameters(cnn_model)))

In [15]:
#| eval: false
LEARNING_RATE = 5e-4 # adjustable parameter (default=5e-4)

# loss function
criterion = nn.CrossEntropyLoss().to(DEVICE)

# optimizer
optimizer = optim.Adam(cnn_model.parameters(), lr=LEARNING_RATE)

In [16]:
#| eval: false
N_EPOCHS = 10 # adjustable parameter (default=10)

# train model for N_EPOCHS epochs
train_model(cnn_model, N_EPOCHS, train_loader, optimizer, criterion)

In [17]:
#| eval: false
def evaluate(model, data_loader, criterion, use_tqdm=False):
  print('Evaluating performance on the test dataset...')
  model.eval()
  epoch_loss = 0
  epoch_acc = 0
  all_predictions = []
  print("\nSOME PREDICTIONS FROM THE MODEL:")
  iterator = tqdm(data_loader) if use_tqdm else data_loader
  total = 0
  for texts, labels in iterator:
    bs = texts.shape[0]
    total += bs
    texts = texts.to(DEVICE)
    labels = labels.to(DEVICE)
    
    output = model(texts)
    acc = accuracy(output, labels) * len(labels)
    pred = output.argmax(dim=1)
    all_predictions.append(pred)
    
    loss = criterion(output, labels) * len(labels)
    
    epoch_loss += loss.item()
    epoch_acc += acc.item()

    if random.random() < 0.0015 and bs == 1:
      print("Prediction:", pred.item(), '\tCorrect Output:', labels.item())
      print("Input: "+' '.join([data_loader.dataset.idx2word[idx] for idx in texts[0].tolist() if idx not in {data_loader.dataset.word2idx[PAD], data_loader.dataset.word2idx[END]}]), '\n')

  full_acc = 100*epoch_acc/total
  full_loss = epoch_loss/total
  print('[TEST]\t Loss: {:.4f}\t Accuracy: {:.2f}%'.format(full_loss, full_acc))
  predictions = torch.cat(all_predictions)
  return predictions, full_acc, full_loss

In [18]:
#| eval: false
evaluation = evaluate(cnn_model, test_loader, criterion, use_tqdm=True) # compute test data accuracy

In [19]:
#| eval: false
class RNN(nn.Module):
  def __init__(self, vocab_size, embed_size, hidden_size, num_layers, bidirectional, dropout, num_classes, pad_idx):
    super(RNN, self).__init__()

    self.bidirectional = bidirectional

    # embedding layer
    self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_size, padding_idx=pad_idx)
    # recurrent network (GRU)
    self.rnn = nn.GRU(input_size=embed_size, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout, batch_first=True, bidirectional=bidirectional)
    # dropout layer
    self.dropout = nn.Dropout(dropout)
    # linear layer
    B = 2 if bidirectional else 1
    self.linear = nn.Linear(B*hidden_size, num_classes)

  def forward(self, texts):
    # embed texts -- [batch_size, max_len, embed_size]
    x = self.embedding(texts)
    # rnn layer
    x, hn = self.rnn(x)
    # concatenate the outputs of the last timestep for each direction -- [batch_size, num_dirs*hidden_size]
    x = torch.cat( (hn[-2, :, :], hn[-1, :, :]), dim=1 ) if self.bidirectional else hn[-1, :, :]
    # apply dropout
    x = self.dropout(x)
    # pass output through the linear layer -- [batch_size, num_classes]
    x = self.linear(x)

    # softmax is applied in training
    return x

In [20]:
#| eval: false
# adjustable parameters (defaults: THRESHOLD=5, MAX_LEN=200, BATCH_SIZE=32)
THRESHOLD = 5
MAX_LEN = 200
BATCH_SIZE = 32

train_dataset = TextDataset(train_data, 'train', THRESHOLD, MAX_LEN)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
print('train vocab size:', train_dataset.vocab_size)

test_dataset = TextDataset(test_data, 'test', THRESHOLD, MAX_LEN, train_dataset.idx2word, train_dataset.word2idx)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=1, drop_last=False)
print('test vocab size:', test_dataset.vocab_size)

In [21]:
#| eval: false
rnn_model = RNN(vocab_size = train_dataset.vocab_size,
            embed_size = 128, 
            hidden_size = 128, 
            num_layers = 2,
            bidirectional = True,
            dropout = 0.5,
            num_classes = 2,
            pad_idx = train_dataset.word2idx[PAD])

# load model on device
rnn_model = rnn_model.to(DEVICE)

print('The model has {:,d} trainable parameters'.format(count_parameters(rnn_model)))

In [22]:
#| eval: false
LEARNING_RATE = 5e-4 # adjustable parameter (default=5e-4)

# loss function
criterion = nn.CrossEntropyLoss().to(DEVICE)

# optimizer
optimizer = optim.Adam(rnn_model.parameters(), lr=LEARNING_RATE)

In [23]:
#| eval: false
N_EPOCHS = 10 # adjustable parameter (default=6)

# train model for N_EPOCHS epochs
train_model(rnn_model, N_EPOCHS, train_loader, optimizer, criterion)

In [24]:
#| eval: false
evaluation = evaluate(rnn_model, test_loader, criterion, use_tqdm=True) # compute test data accuracy